# Análise de Casos de Dengue no Segundo Semestre

Este notebook contém:
- carregamento e preparação dos dados;
- definição dos cenários ENSO;
- agregação espacial dos dados;
- análises exploratórias;
- visualizações por regional e município;
- análise da próxima temporada epidemiológica.


## 1. Importação de bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt 
import matplotlib.gridspec as gridspec

In [ ]:
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.titlesize': 16
})

## 2. Definição de estados e cenários ENSO

In [ ]:
states_BR = ['AL',
 'BA',
 'CE',
 'MA',
 'PB',
 'PE',
 'PI',
 'SE',
 'RN',
 'SP',
 'MG',
 'RJ',
 'ES',
 'AM',
 'AP',
 'TO',
 'RR',
 'RO',
 'AC',
 'PA',
 'DF',
 'GO',
 'MT',
 'MS',
 'RS',
 'SC',
 'PR']

In [ ]:
el_nino_2_sem = [2014, 2015, 2018, 2023]
neutros_2_sem = [2012, 2013, 2019, 2024]

## 3. Dados regionais de saúde

Regional de saúde: 

In [ ]:
df_map = pd.read_csv('../imdc_2026/data/map_regional_health.csv')

df_map.head()

In [ ]:
list_df_agg = [] 

for state  in states_BR: 
    df_data = pd.read_parquet(f'../load_infodengue_data/data/cases/{state}_dengue.parquet',
                            columns = ['data_iniSE', 'casos', 'casos_est', 'municipio_geocodigo', 'pop'])
    

    df_data = df_data.loc[df_data['pop'] >=100000]


    df_data = df_data.reset_index().merge(df_map[['regional_geocode', 'geocode', 'macroregion_name', 'uf']], left_on = 'municipio_geocodigo', 
                            right_on = 'geocode').set_index('data_iniSE')  


    df_data['year'] = df_data.index.year

    df_data['semester'] = 1

    df_data.loc[df_data.index.month >= 7, 'semester'] = 2 

    df_agg = df_data.groupby(['macroregion_name', 'uf',  'geocode', 'year', 'semester'])[['casos']].sum()

    list_df_agg.append(df_agg)


df_agg_end = pd.concat(list_df_agg)

df_agg_end = df_agg_end.reset_index()

df_agg_end.head()

## 4. Visualizações agregadas

In [ ]:
fig = plt.figure(figsize=(16, 8))
gs = gridspec.GridSpec(2, 6, figure=fig)

# First row with three boxplots
ax1 = fig.add_subplot(gs[0, 0:2])
ax2 = fig.add_subplot(gs[0, 2:4])
ax3 = fig.add_subplot(gs[0, 4:6])

# Second row with two boxplots
ax4 = fig.add_subplot(gs[1, 1:3])
ax5 = fig.add_subplot(gs[1, 3:5])


for r, ax in zip(['GO', 'MG', 'PR', 'AM', 'CE'], [ax1,ax2,ax3,ax4,ax5]):

    df_ = df_agg_end.loc[(df_agg_end.uf == r) & (df_agg_end.semester == 2)]

    df_ = df_.groupby(['year'])[['casos']].sum()

    colors = [
        'red' if year in el_nino_2_sem else 'steelblue'
        for year in df_.index
    ]

    ax.bar(df_.index, df_.casos, color=colors)

    ax.set_title(r)
    ax.grid(axis = 'y')


plt.suptitle('Soma dos casos no 2º semestre\nAs barras vermelhas representam os anos de el niño')
# Adjust layout
plt.tight_layout()
plt.savefig('figures/soma_2_semestre_uf.png', dpi = 200, bbox_inches = 'tight')
plt.show()

## 5. Casos no segundo semestre por cidade

### Casos no segundo semeste - gráfico de violino por cidade

In [ ]:
df_agg_end.loc[:, 'cenario'] = None

df_agg_end = df_agg_end.loc[df_agg_end.year >= 2018]

df_agg_end.loc[df_agg_end.year.isin(el_nino_2_sem), 'cenario'] = 'El niño' 
df_agg_end.loc[df_agg_end.year.isin(neutros_2_sem), 'cenario'] = 'Neutro' 

df_agg_end = df_agg_end.dropna()

df_agg_end.year.value_counts()

In [ ]:
fig = plt.figure(figsize=(16, 8))
gs = gridspec.GridSpec(2, 6, figure=fig)

# First row with three boxplots
ax1 = fig.add_subplot(gs[0, 0:2])
ax2 = fig.add_subplot(gs[0, 2:4])
ax3 = fig.add_subplot(gs[0, 4:6])

# Second row with two boxplots
ax4 = fig.add_subplot(gs[1, 1:3])
ax5 = fig.add_subplot(gs[1, 3:5])


for r, ax in zip(['Sul', 'Sudeste', 'Centro-Oeste', 'Norte', 'Nordeste'], [ax1,ax2,ax3,ax4,ax5]):

    df_ = df_agg_end.loc[(df_agg_end.macroregion_name == r) & (df_agg_end.semester == 2)]

    #df_ = df_.loc[df_.casos > 10 ]

    outros = df_.loc[df_.cenario == 'Neutro'].casos.values

    el_nino = df_.loc[df_.cenario == 'El niño'].casos.values

    p = stats.ttest_ind(outros,el_nino, alternative = 'two-sided').pvalue

    #print(p)

    if p < 0.01:
        label = '(p < 0.01)'
    if p >= 0.01:
        label = '(p > 0.01)'

    ax.set_title(f'{r} {label}', fontsize = 16)

    sns.violinplot(data = df_, x = 'cenario', y = 'casos', palette = ['tab:blue', 'tab:red'],  hue = 'cenario', legend = False, ax=ax)
    
    ax.axhline(np.median(el_nino), color = 'orange', zorder = 2,  label = 'Mediana - El niño')
    
    ax.set_xlabel('')

    ax.legend()
    ax.set_yscale('log')
    ax.grid(axis = 'y')


plt.suptitle('Soma dos casos no 2º semestre.\n El niño: 2018 e 2023, Neutro: 2019 e 2024.')
# Adjust layout
plt.tight_layout()
plt.savefig('figures/soma_2_semestre_region_el_nino_neutro.png', dpi = 200, bbox_inches = 'tight')
plt.show()

## 6. Análise da próxima temporada

### Próxima temporada: 

In [ ]:
el_nino_pos = [2015, 2016, 2019, 2024]

In [ ]:
df_agg_y = df_agg_end.groupby(['macroregion_name', 'uf', 'geocode', 'year'])[['casos']].sum().reset_index()


df_agg_y = df_agg_y.loc[df_agg_y.casos > 10]

df_agg_y['el_nino'] = False

df_agg_y.loc[df_agg_y.year.isin(el_nino_pos), 'el_nino'] = True

df_agg_y.head()

In [ ]:
fig = plt.figure(figsize=(16, 8))
gs = gridspec.GridSpec(2, 6, figure=fig)

# First row with three boxplots
ax1 = fig.add_subplot(gs[0, 0:2])
ax2 = fig.add_subplot(gs[0, 2:4])
ax3 = fig.add_subplot(gs[0, 4:6])

# Second row with two boxplots
ax4 = fig.add_subplot(gs[1, 1:3])
ax5 = fig.add_subplot(gs[1, 3:5])


for r, ax in zip(['Sul', 'Sudeste', 'Centro-Oeste', 'Norte', 'Nordeste'], [ax1,ax2,ax3,ax4,ax5]):

    df_ = df_agg_y.loc[(df_agg_y.macroregion_name == r)]

    outros = df_.loc[df_.el_nino == False].casos.values

    el_nino = df_.loc[df_.el_nino == True].casos.values

    p = stats.ttest_ind(outros,el_nino, alternative = 'two-sided').pvalue

    #print(p)

    if p < 0.01:
        label = '(p < 0.01)'
    if p >= 0.01:
        label = '(p > 0.01)'

    ax.set_xlabel('El Niño')

    ax.set_title(f'{r} {label}', fontsize = 16)

    sns.violinplot(data = df_, x = 'el_nino', y = 'casos',  palette = ['tab:blue', 'tab:red'],  hue = 'el_nino', legend = False, ax=ax)

    ax.axhline(np.median(el_nino), color = 'orange', zorder = 2,  label = 'Mediana - El niño')

    ax.legend()
    ax.set_yscale('log')
    ax.grid(axis = 'y')


plt.suptitle('Soma dos casos por ano.\nO valor true se refere aos anos de El Niño.')
# Adjust layout
plt.tight_layout()
plt.savefig('figures/soma_ano_el_nino_region.png', dpi = 200, bbox_inches = 'tight')
plt.show()